## Prepare Google Colab Environment

In [ ]:
import os

#@title Install Packages {display-mode:"form"}
INSTALL_PACKAGES = True #@param {type:"boolean"}

# Check if packages have already been installed in this session to prevent re-installation
if INSTALL_PACKAGES and not os.environ.get('PYPSA_PACKAGES_INSTALLED'):
  !pip install pypsa pypsa[excel] folium mapclassify cartopy
  os.environ['PYPSA_PACKAGES_INSTALLED'] = 'true'
elif not INSTALL_PACKAGES:
  print("Skipping package installation.")
else:
  print("PyPSA packages are already installed for this session.")

# Exercise 1.1

**Objective:**

Programatically build a PyPSA network, define the model’s constraints, solve it, and review the results.

**Diagram**

<img src="https://raw.githubusercontent.com/PriyeshGosai/pypsa-meets-earth-lab-2025/main/img/example_1_img.png" alt="Example 1" width="50%">

In [ ]:
import pypsa
import pandas as pd
import numpy as np
pypsa.options.api.new_components_api = True

snapshots_df = pd.date_range('2025-01-01 00:00', '2025-12-31 23:00', freq='h')
load_profile = np.random.rand(len(snapshots_df)) * 100

n = pypsa.Network()
n.set_snapshots(snapshots_df)
n.add('Carrier',['gas','AC'])
n.add('Bus','Location',carrier = 'AC')

n.add('Load',
      'Load A',
      bus = 'Location',
      p_set = load_profile,
      carrier = 'gas')

n.add('Generator',
      'Generator A',
      bus = 'Location',
      p_nom = 10,
      marginal_cost = 1,
      p_nom_extendable = True)



In [ ]:
n.generators.static.p_nom_opt

In [ ]:
n.generators.static

In [ ]:
n.loads.static

In [ ]:
n.buses.static

In [ ]:
n.optimize()

In [ ]:
n.generators.static

In [ ]:
n.objective_constant

In [ ]:
n.buses.dynamic.marginal_price.sum()

In [ ]:
n.snapshot_weightings

In [ ]:
n.objective

In [ ]:
for key in n.buses.dynamic:
    print(key)

In [ ]:
n.loads

In [ ]:
n.objective

# Exercise 1.2

In this example, we will use an example network distributed with PyPSA to observe more complex features. 



## Meshed AC–DC Network Optimisation in PyPSA

This example demonstrates how to optimise a **meshed AC–DC network** in PyPSA.  
The network contains a **3-node AC system** connected via **AC–DC converters** to a **3-node DC system**.  
There is also a **point-to-point DC connection** represented using the `Link` component.

Reference example:  
https://docs.pypsa.org/latest/examples/ac-dc-lopf/


In [ ]:
import pypsa
pypsa.options.api.new_components_api = True

# Fix for pandas 2.3.0 StringDtype incompatibility with PyPSA plotting
import pandas as pd
pd.options.future.infer_string = False

network = pypsa.examples.ac_dc_meshed()

In [ ]:
network.carriers.static

In [ ]:
network.carriers.static

In [ ]:
network.global_constraints.static

In [ ]:
network.generators.static.efficiency

In [ ]:
network.generators.static.p_nom_extendable

In [ ]:
network.generators.static

In [ ]:
# line_color = network.lines.static.bus0.map(network.buses.static.carrier).map(
#     lambda ct: "r" if ct == "DC" else "b"
# ).astype(object) 

# network.plot.explore(
#     # line_color=line_color,
#     link_color="c",
#     jitter=0.4,
# )

In [ ]:
network.determine_network_topology()


In [ ]:
network.snapshots

In [ ]:
network.buses.static

In [ ]:
network.generators.static

In [ ]:
network.generators.dynamic.p_max_pu

In [ ]:
network.lines.static

In [ ]:
network.links.static

In [ ]:
network.loads.static

In [ ]:
network.loads.dynamic.p_set

In [ ]:
network.loads.dynamic.p_set.plot()

In [ ]:
network.global_constraints.static

In [ ]:
network.sub_networks.static

In [ ]:
network.sub_networks.static.loc['0','obj']

In [ ]:
network.sub_networks.static.loc['0','obj'].components.buses.static

Solve the model

In [ ]:
network.optimize()

View all the constraints

In [ ]:
network.model

View Results

In [ ]:
network.generators.dynamic.p.plot()

In [ ]:
network.links.dynamic.p0.plot()

In [ ]:
network.lines.dynamic.p0.plot()

Export the results file